# ITRI external validation: protocol v2 amendment (EXPLORATORY), per-window normalization

**Runtime:** GPU, tercihen **A100**. High RAM **gerekmez** (veri kucuk, modeller kucuk). Sonra **Run all**.
Istendiginde **yalnizca** `ITRI_external_PRIVATE_data_snapshot.zip` dosyasini secin (SHA-256 dogrulanir). Drive baglanir;
sadece metrik kontrol noktalari Drive'a yazilir, ham OHLCV yazilmaz.

Bu surum **SAITS V1'i atlar** (30 epoch tam izgara zaten CPU'da tamamlandi ve makaleye islendi). Sirayla:

1. **SAITS V2** (200 epoch, patience 20), tam izgara 5.508 kosul. A100'de kosul basina ~4-6 s beklenir, toplam ~6-9 saat.
2. **BRITS V1** (30 epoch), yalnizca **fold A + seed 11 alt kumesi** (612 kosul). BRITS PyPOTS'ta zaman adimi dongusu yuzunden
   GPU'da hizlanmaz (A100'de ~45 s/kosul); tam izgara 70 saati asar, bu yuzden etiketli alt kume. ~8 saat.

Oturum kesilirse ayni notebook'u tekrar **Run all** yapin: tamamlanmis varlik/fold parcalari SHA-256 ile dogrulanip atlanir,
kalan yerden devam edilir. Sonunda `ITRI_neural_windownorm_results.zip` otomatik iner. Once sadece SAITS V2 isteniyorsa
asagidaki `RUNS` listesinden BRITS satirini silin.

Bu kosu **kesifsel** bir protokol ekidir (v2); v1 birincil sonucun yerine gecmez.

In [ ]:
%pip -q install pypots==1.5 numpy==2.1.3 pandas==2.2.3 scipy==1.18.1 scikit-learn==1.6.1

In [ ]:
from pathlib import Path
work_dir = Path('/content/itri_neural')
work_dir.mkdir(parents=True, exist_ok=True)
print('Working directory:', work_dir)

In [ ]:
%%writefile /content/itri_neural/masking.py
"""Controlled missing-run masks for a new, separately reported validation study.

The archived ITRI experiment permits adjacent sampled segments. This module
does not alter those historical masks or results. It defines strict runs for a
future experiment in which the requested block length has a literal meaning.
"""

from __future__ import annotations

import numpy as np


def run_lengths(mask: np.ndarray) -> np.ndarray:
    """Return every maximal missing-run length, separately for each feature."""
    if mask.ndim != 2 or mask.dtype != np.bool_:
        raise ValueError("mask must be a two-dimensional boolean array")
    lengths: list[int] = []
    for feature in range(mask.shape[1]):
        edges = np.flatnonzero(np.diff(np.r_[False, mask[:, feature], False]))
        lengths.extend((edges[1::2] - edges[::2]).tolist())
    return np.asarray(lengths, dtype=int)


def strict_block_mask(
    shape: tuple[int, int], rate: float, block_length: int, seed: int,
    edge_guard: int = 10,
) -> np.ndarray:
    """Place nonadjacent runs of length ``block_length`` at an exact cell rate.

    Every feature receives one full run before the remaining runs are sampled,
    so feature-wise errors are defined for every condition. A final shorter
    run is allowed only when the rounded target cell count is not divisible
    by the requested block length. One observed row separates every pair of
    runs within a feature. Other features may be missing on the same dates.
    """
    rows, features = shape
    if rows <= 2 * edge_guard + block_length or features < 1:
        raise ValueError("not enough rows/features for the block and guard")
    if not 0 < rate < 1 or block_length < 1 or edge_guard < 0:
        raise ValueError("invalid rate, block length, or edge guard")
    eligible_rows = rows - 2 * edge_guard
    target = max(1, round(rate * eligible_rows * features))
    if target < features * block_length:
        raise ValueError("target too small for one full masked run per feature")
    rng = np.random.default_rng(seed)
    mask = np.zeros(shape, dtype=bool)
    # The endpoint is a five-feature macro mean, so a zero-mask feature is
    # invalid. Reserve exactly one nominal-length run per feature first.
    for feature in rng.permutation(features):
        start = int(rng.integers(edge_guard,
                                 rows - edge_guard - block_length + 1))
        mask[start:start + block_length, feature] = True
    remaining = target - features * block_length
    attempts = 0
    max_attempts = 1000 * target

    while remaining:
        length = min(block_length, remaining)
        feature = int(rng.integers(features))
        start = int(rng.integers(edge_guard, rows - edge_guard - length + 1))
        neighborhood = mask[max(edge_guard, start - 1):
                            min(rows - edge_guard, start + length + 1), feature]
        if not neighborhood.any():
            mask[start:start + length, feature] = True
            remaining -= length
        attempts += 1
        if attempts >= max_attempts:
            raise RuntimeError("could not place the target runs without adjacency")

    lengths = run_lengths(mask)
    if (int(mask.sum()) != target or lengths.max() > block_length or
            not np.all(mask.sum(axis=0) >= block_length)):
        raise AssertionError("strict-block mask invariant failed")
    if np.any(mask[:edge_guard]) or np.any(mask[rows - edge_guard:]):
        raise AssertionError("strict-block mask crossed the edge guard")
    return mask


In [ ]:
%%writefile /content/itri_neural/run_classical_benchmark.py
#!/usr/bin/env python3
"""Frozen OHLCV external-validation benchmark, classical comparators.

This program reads the PRIVATE vendor snapshot ZIP without extracting it. It
never writes OHLCV values to the public result directory. All methods receive
the exact same mask per asset/fold/condition. Outputs are checkpointed by
asset/fold. Use --pilot for a small smoke test; never report pilot results as
the full prespecified experiment.
"""

from __future__ import annotations

import argparse
import hashlib
import io
import json
import os
import platform
import time
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy.interpolate import PchipInterpolator
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, KNNImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.exceptions import ConvergenceWarning

from masking import run_lengths, strict_block_mask


FEATURES = ("open", "high", "low", "close", "volume")
METHODS = ("linear", "pchip", "knn5", "chained_ridge", "itri_w1_p5")
FOLDS = {
    "A": {"train": (2015, 2018), "val": (2019, 2019), "test": (2020, 2021)},
    "B": {"train": (2015, 2020), "val": (2021, 2021), "test": (2022, 2023)},
    "C": {"train": (2015, 2022), "val": (2023, 2023), "test": (2024, 2025)},
}
BLOCKS = (1, 2, 3, 5, 10, 20)
RATES = (0.10, 0.20, 0.30)
SEEDS = (11, 23, 37)
EDGE_GUARD = 20


def sha256(blob: bytes) -> str:
    return hashlib.sha256(blob).hexdigest()


def stable_mask_seed(manifest_sha: str, ticker: str, fold: str,
                     partition: str, block: int, rate: float,
                     repetition_seed: int) -> int:
    key = (f"{manifest_sha}|{ticker}|{fold}|{partition}|"
           f"{block}|{rate:.2f}|{repetition_seed}")
    return int.from_bytes(hashlib.sha256(key.encode()).digest()[:4], "big")


def split_frame(frame: pd.DataFrame, fold: str) -> dict[str, pd.DataFrame]:
    parts = {}
    for name, (start_year, end_year) in FOLDS[fold].items():
        subset = frame.loc[(frame.index.year >= start_year) &
                           (frame.index.year <= end_year), list(FEATURES)]
        if subset.empty:
            raise ValueError(f"empty {fold}/{name} partition")
        parts[name] = subset.copy()
    return parts


def train_observed_scaler(train: np.ndarray, hidden: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    if train.shape != hidden.shape:
        raise ValueError("training array/mask shape mismatch")
    means, scales = [], []
    for feature in range(train.shape[1]):
        observed = train[~hidden[:, feature], feature]
        if observed.size < 2 or not np.isfinite(observed).all():
            raise ValueError(f"insufficient finite training data for feature {feature}")
        mean = float(observed.mean())
        scale = float(observed.std(ddof=0))
        if scale <= 1e-12:
            raise ValueError(f"near-zero training-observed std for feature {feature}")
        means.append(mean)
        scales.append(scale)
    return np.asarray(means), np.asarray(scales)


def corrupt(values: np.ndarray, hidden: np.ndarray) -> np.ndarray:
    output = values.copy()
    output[hidden] = np.nan
    return output


def linear_fill(corrupt_values: np.ndarray) -> np.ndarray:
    return pd.DataFrame(corrupt_values).interpolate(
        method="linear", limit_direction="both").ffill().bfill().to_numpy(float)


def pchip_fill(corrupt_values: np.ndarray) -> np.ndarray:
    out = corrupt_values.copy()
    x = np.arange(len(out))
    for feature in range(out.shape[1]):
        seen = np.isfinite(out[:, feature])
        if seen.sum() < 2:
            raise ValueError("PCHIP needs at least two observed cells")
        curve = PchipInterpolator(x[seen], out[seen, feature], extrapolate=True)
        out[~seen, feature] = curve(x[~seen])
    return out


class ITRIImputer:
    """Exact w=1, p=5 temporal-regression logic from the archived implementation.

    Only the panel and observed-only scaling differ from the original study.
    The base model has not been modified to chase external test results.
    """

    def __init__(self) -> None:
        self.window = 1
        self.models_: list[LinearRegression | None] = []

    def temporal_matrix(self, col: np.ndarray) -> np.ndarray:
        z = np.column_stack((np.roll(col, 1), np.roll(col, -1)))
        z[0, :] = np.nan
        z[-1, :] = np.nan
        return z

    def fit(self, train_corrupt: np.ndarray) -> "ITRIImputer":
        missing = np.isnan(train_corrupt)
        completed = linear_fill(train_corrupt)
        self.models_ = []
        for feature in range(train_corrupt.shape[1]):
            z = self.temporal_matrix(completed[:, feature])
            good = (~missing[:, feature]) & np.isfinite(z).all(axis=1)
            if int(good.sum()) < 4:
                self.models_.append(None)
                continue
            model = LinearRegression()
            model.fit(z[good], completed[good, feature])
            self.models_.append(model)
        return self

    def transform(self, test_corrupt: np.ndarray) -> np.ndarray:
        missing = np.isnan(test_corrupt)
        observed = ~missing
        completed = linear_fill(test_corrupt)
        for _ in range(5):
            for feature, model in enumerate(self.models_):
                if model is None:
                    continue
                z = self.temporal_matrix(completed[:, feature])
                rows = missing[:, feature] & np.isfinite(z).all(axis=1)
                if rows.any():
                    completed[rows, feature] = model.predict(z[rows])
            completed[observed] = test_corrupt[observed]
        return completed


def run_methods(train_corrupt: np.ndarray, test_corrupt: np.ndarray,
                random_seed: int) -> dict[str, tuple[np.ndarray, float, int]]:
    outputs = {}
    start = time.perf_counter()
    outputs["linear"] = (linear_fill(test_corrupt), time.perf_counter() - start, 0)
    start = time.perf_counter()
    outputs["pchip"] = (pchip_fill(test_corrupt), time.perf_counter() - start, 0)
    start = time.perf_counter()
    knn = KNNImputer(n_neighbors=5, weights="distance")
    knn.fit(train_corrupt)
    outputs["knn5"] = (knn.transform(test_corrupt), time.perf_counter() - start, 0)
    start = time.perf_counter()
    chained = IterativeImputer(
        estimator=Ridge(alpha=1.0), max_iter=10, tol=1e-3,
        initial_strategy="mean", imputation_order="ascending",
        sample_posterior=False, random_state=random_seed,
    )
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        chained.fit(train_corrupt)
        chained_estimate = chained.transform(test_corrupt)
    nonconvergence = sum(issubclass(note.category, ConvergenceWarning)
                         for note in caught)
    outputs["chained_ridge"] = (chained_estimate,
                                 time.perf_counter() - start, nonconvergence)
    start = time.perf_counter()
    temporal = ITRIImputer().fit(train_corrupt)
    outputs["itri_w1_p5"] = (temporal.transform(test_corrupt),
                               time.perf_counter() - start, 0)
    if tuple(outputs) != METHODS:
        raise AssertionError("method registry/order mismatch")
    for method, (estimate, _, _) in outputs.items():
        if estimate.shape != test_corrupt.shape or not np.isfinite(estimate).all():
            raise ValueError(f"{method} produced invalid estimates")
        observed = np.isfinite(test_corrupt)
        if not np.allclose(estimate[observed], test_corrupt[observed],
                           atol=1e-9, rtol=1e-9):
            raise ValueError(f"{method} changed originally observed cells")
    return outputs


def score_features(truth_raw: np.ndarray, estimate_scaled: np.ndarray,
                   hidden: np.ndarray, mean: np.ndarray, scale: np.ndarray,
                   common: dict, elapsed: float,
                   convergence_warning_count: int) -> list[dict]:
    estimate_raw = estimate_scaled * scale + mean
    results = []
    for feature_index, feature in enumerate(FEATURES):
        selected = hidden[:, feature_index]
        count = int(selected.sum())
        if count < 1:
            raise ValueError(f"no masked cells for {feature}")
        error = estimate_raw[selected, feature_index] - truth_raw[selected, feature_index]
        mae = float(np.abs(error).mean())
        rmse = float(np.sqrt(np.square(error).mean()))
        results.append({
            **common,
            "feature": feature,
            "masked_cells": count,
            "mae_raw": mae,
            "rmse_raw": rmse,
            "mae_trainstd": mae / scale[feature_index],
            "rmse_trainstd": rmse / scale[feature_index],
            "fit_and_impute_seconds": elapsed,
            "convergence_warning_count": convergence_warning_count,
        })
    return results


def load_verified_asset(archive: zipfile.ZipFile, record: dict) -> pd.DataFrame:
    name = f"clean_vendor_cache_do_not_publish/{record['ticker']}_2015-01-01_2026-01-01.csv"
    blob = archive.read(name)
    if sha256(blob) != record["clean_sha256"]:
        raise ValueError(f"private data hash mismatch: {record['ticker']}")
    frame = pd.read_csv(io.BytesIO(blob), index_col=0,
                        parse_dates=True, float_precision="round_trip")
    if frame.columns.tolist() != list(FEATURES) or len(frame) != record["row_count"]:
        raise ValueError(f"private data shape mismatch: {record['ticker']}")
    if not np.isfinite(frame.to_numpy(float)).all():
        raise ValueError(f"nonfinite data: {record['ticker']}")
    return frame


def run(snapshot: Path, output_dir: Path, pilot: bool = False) -> None:
    with zipfile.ZipFile(snapshot) as archive:
        manifest_blob = archive.read("asset_universe_v1.json")
        receipt_blob = archive.read("acquisition_qc_receipt.json")
        receipt = json.loads(receipt_blob)
        manifest = json.loads(manifest_blob)
        manifest_sha = sha256(manifest_blob)
        if receipt["manifest_sha256"] != manifest_sha:
            raise ValueError("snapshot manifest/receipt SHA mismatch")
        if receipt["eligible_count"] != 34 or receipt["excluded_count"]:
            raise ValueError("unexpected eligibility count; revise only by protocol amendment")
        if [x["ticker"] for x in manifest["assets"]] != [
                x["ticker"] for x in receipt["assets"]]:
            raise ValueError("manifest/receipt asset order mismatch")

        output_dir.mkdir(parents=True, exist_ok=True)
        shard_dir = output_dir / "shards"
        shard_dir.mkdir(exist_ok=True)
        versions = {"python": platform.python_version(), "numpy": np.__version__,
                    "pandas": pd.__version__, "scipy": scipy.__version__,
                    "sklearn": sklearn.__version__}
        run_fingerprint = sha256(
            Path(__file__).read_bytes() + Path(__file__).with_name("masking.py").read_bytes()
            + manifest_blob + receipt_blob + str(pilot).encode()
            + json.dumps(versions, sort_keys=True).encode())
        run_meta = {
            "run_fingerprint": run_fingerprint,
            "protocol": manifest["protocol_version"],
            "manifest_sha256": manifest_sha,
            "private_zip_sha256": sha256(snapshot.read_bytes()),
            "pilot_not_publishable": pilot,
            "methods": METHODS,
            "folds": FOLDS,
            "blocks": BLOCKS,
            "rates": RATES,
            "seeds": SEEDS,
            "edge_guard": EDGE_GUARD,
            "primary_panel": FEATURES,
            "primary_endpoint": "macro mean across 5 feature-wise RMSE/training-observed-SD",
            "versions": versions,
        }
        (output_dir / "run_metadata.json").write_text(
            json.dumps(run_meta, indent=2), encoding="utf-8")
        records = receipt["assets"][:1] if pilot else receipt["assets"]
        folds = ("A",) if pilot else tuple(FOLDS)
        conditions = [(b, r, s) for b in BLOCKS for r in RATES for s in SEEDS]
        if pilot:
            conditions = conditions[:2]
        for asset_index, record in enumerate(records, start=1):
            ticker = record["ticker"]
            frame = load_verified_asset(archive, record)
            for fold in folds:
                shard_path = shard_dir / f"{ticker}_{fold}.csv"
                meta_path = shard_dir / f"{ticker}_{fold}.meta.json"
                expected_rows = len(conditions) * len(METHODS) * len(FEATURES)
                if shard_path.exists() and meta_path.exists():
                    sidecar = json.loads(meta_path.read_text())
                    if (sidecar.get("run_fingerprint") == run_fingerprint and
                            sidecar.get("expected_rows") == expected_rows and
                            sidecar.get("csv_sha256") == sha256(shard_path.read_bytes())):
                        print(f"[{asset_index}/{len(records)}] {ticker}/{fold}: verified checkpoint")
                        continue
                partitions = split_frame(frame, fold)
                train = partitions["train"].to_numpy(float)
                test = partitions["test"].to_numpy(float)
                rows = []
                for condition_index, (block, rate, repetition_seed) in enumerate(conditions, 1):
                    train_seed = stable_mask_seed(manifest_sha, ticker, fold, "train",
                                                  block, rate, repetition_seed)
                    test_seed = stable_mask_seed(manifest_sha, ticker, fold, "test",
                                                 block, rate, repetition_seed)
                    train_hidden = strict_block_mask(train.shape, rate, block,
                                                     train_seed, EDGE_GUARD)
                    test_hidden = strict_block_mask(test.shape, rate, block,
                                                    test_seed, EDGE_GUARD)
                    mean, scale = train_observed_scaler(train, train_hidden)
                    train_scaled = (train - mean) / scale
                    test_scaled = (test - mean) / scale
                    estimates = run_methods(corrupt(train_scaled, train_hidden),
                                            corrupt(test_scaled, test_hidden),
                                            repetition_seed)
                    lengths = run_lengths(test_hidden)
                    common = {
                        "ticker": ticker, "country": record["country"],
                        "region": record["region"], "fold": fold,
                        "test_start_year": FOLDS[fold]["test"][0],
                        "test_end_year": FOLDS[fold]["test"][1],
                        "block_length": block, "rate": rate,
                        "seed": repetition_seed,
                        "train_mask_seed": train_seed,
                        "test_mask_seed": test_seed,
                        "actual_max_test_run": int(lengths.max()),
                    }
                    if common["actual_max_test_run"] > block:
                        raise AssertionError("mask-length invariant violated")
                    for method, (estimate, elapsed, warning_count) in estimates.items():
                        rows.extend(score_features(test, estimate, test_hidden,
                                                   mean, scale,
                                                   {**common, "method": method},
                                                   elapsed, warning_count))
                    if condition_index % 10 == 0 or condition_index == len(conditions):
                        print(f"[{asset_index}/{len(records)}] {ticker}/{fold}: "
                              f"{condition_index}/{len(conditions)} conditions")
                if len(rows) != expected_rows:
                    raise AssertionError("missing or duplicate method-feature rows")
                temporary = shard_path.with_suffix(".csv.tmp")
                pd.DataFrame(rows).to_csv(temporary, index=False, float_format="%.17g")
                os.replace(temporary, shard_path)
                sidecar = {
                    "run_fingerprint": run_fingerprint,
                    "expected_rows": expected_rows,
                    "csv_sha256": sha256(shard_path.read_bytes()),
                    "pilot_not_publishable": pilot,
                }
                meta_path.write_text(json.dumps(sidecar, indent=2), encoding="utf-8")
        expected_shards = len(records) * len(folds)
        verified_shards = list(shard_dir.glob("*.meta.json"))
        if len(verified_shards) < expected_shards:
            raise AssertionError("run ended with missing checkpoints")
        (output_dir / "COMPLETE.marker").write_text(
            json.dumps({"expected_shards": expected_shards,
                        "expected_rows": expected_shards * len(conditions)
                        * len(METHODS) * len(FEATURES),
                        "pilot_not_publishable": pilot,
                        "run_fingerprint": run_fingerprint}, indent=2),
            encoding="utf-8")
        print("COMPLETE", output_dir)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--snapshot", type=Path, required=True,
                        help="Private data snapshot ZIP from acquisition notebook")
    parser.add_argument("--out", type=Path, required=True)
    parser.add_argument("--pilot", action="store_true",
                        help="Smoke test only: first asset/fold/two masks; not publishable")
    args = parser.parse_args()
    run(args.snapshot, args.out, pilot=args.pilot)


In [ ]:
%%writefile /content/itri_neural/run_neural_benchmark.py
#!/usr/bin/env python3
"""BRITS/SAITS external validation on the frozen classical benchmark masks.

One neural model is fitted for each asset, fold, block length, missing rate,
mask seed and method. Only the corresponding masked training partition is
used for fitting; the separate validation partition selects the epoch. Test
truth is used only by score_features after inference. Checkpoints contain
metrics and metadata, never raw OHLCV or model weights.
"""

from __future__ import annotations

import argparse
import hashlib
import io
import json
import logging
import os
import platform
import random
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import pypots
import pygrinder
import scipy
import sklearn
import torch
from pypots.imputation import BRITS, SAITS
from pypots.utils.logging import logger as pypots_logger

from masking import run_lengths, strict_block_mask
from run_classical_benchmark import (
    BLOCKS, EDGE_GUARD, FEATURES, FOLDS, RATES, SEEDS, METHODS,
    corrupt, load_verified_asset, run_methods, score_features, sha256, split_frame,
    stable_mask_seed, train_observed_scaler,
)


NEURAL_METHODS = ("brits", "saits")
N_STEPS = 64
STRIDE = 16
BATCH_SIZE = 32
EPOCHS = 30
PATIENCE = 5
BRITS_HIDDEN = 64
SAITS_CONFIG = {"n_layers": 2, "d_model": 64, "n_heads": 4,
                "d_k": 16, "d_v": 16, "d_ffn": 128, "dropout": 0.1}


class _NoSavingPathWarning(logging.Filter):
    def filter(self, record: logging.LogRecord) -> bool:
        return "saving_path not given" not in record.getMessage()


pypots_logger.addFilter(_NoSavingPathWarning())


def window_starts(length: int, n_steps: int = N_STEPS,
                  stride: int = STRIDE) -> list[int]:
    if length < n_steps or n_steps < 2 or stride < 1:
        raise ValueError("partition is too short for the frozen neural window")
    starts = list(range(0, length - n_steps + 1, stride))
    if starts[-1] != length - n_steps:
        starts.append(length - n_steps)
    return starts


def as_windows(values: np.ndarray, starts: list[int],
               n_steps: int = N_STEPS) -> np.ndarray:
    return np.stack([values[start:start + n_steps] for start in starts]).astype(
        np.float32, copy=False)


def fold_windows(predictions: np.ndarray, starts: list[int],
                 observed: np.ndarray) -> np.ndarray:
    """Average overlapping predictions; restore every originally observed cell."""
    if predictions.shape != (len(starts), N_STEPS, observed.shape[1]):
        raise ValueError("neural prediction shape mismatch")
    sums = np.zeros(observed.shape, dtype=np.float64)
    counts = np.zeros(observed.shape[0], dtype=np.int32)
    for prediction, start in zip(predictions, starts):
        sums[start:start + N_STEPS] += prediction
        counts[start:start + N_STEPS] += 1
    if np.any(counts == 0):
        raise AssertionError("test window coverage is incomplete")
    completed = sums / counts[:, None]
    originally_observed = np.isfinite(observed)
    completed[originally_observed] = observed[originally_observed]
    if not np.isfinite(completed).all():
        raise ValueError("neural model produced non-finite estimates")
    return completed


def set_training_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def training_seed(manifest_sha: str, ticker: str, fold: str,
                  block: int, rate: float, repetition_seed: int,
                  method: str) -> int:
    key = (f"{manifest_sha}|{ticker}|{fold}|{block}|{rate:.2f}|"
           f"{repetition_seed}|{method}|neural-v1")
    return int.from_bytes(hashlib.sha256(key.encode()).digest()[:4], "big")


def fit_and_predict(method: str, train_input: np.ndarray,
                    val_input: np.ndarray, val_truth: np.ndarray,
                    test_input: np.ndarray, seed: int,
                    device: str, epochs: int = EPOCHS) -> tuple[np.ndarray, np.ndarray, float]:
    set_training_seed(seed)
    train_starts = window_starts(len(train_input))
    val_starts = window_starts(len(val_input))
    test_starts = window_starts(len(test_input))
    train_set = {"X": as_windows(train_input, train_starts)}
    val_set = {"X": as_windows(val_input, val_starts),
               "X_ori": as_windows(val_truth, val_starts)}
    test_set = {"X": as_windows(test_input, test_starts)}
    common = {"n_steps": N_STEPS, "n_features": len(FEATURES),
              "batch_size": BATCH_SIZE, "epochs": epochs,
              "patience": min(PATIENCE, epochs), "device": device,
              "model_saving_strategy": None, "verbose": False}
    if method == "brits":
        model = BRITS(rnn_hidden_size=BRITS_HIDDEN, **common)
    elif method == "saits":
        model = SAITS(**SAITS_CONFIG, **common)
    else:
        raise ValueError(f"unknown neural method: {method}")
    started = time.perf_counter()
    model.fit(train_set, val_set)
    val_predictions = np.asarray(model.predict({"X": val_set["X"]})["imputation"])
    test_predictions = np.asarray(model.predict(test_set)["imputation"])
    val_completed = fold_windows(val_predictions, val_starts, val_input)
    test_completed = fold_windows(test_predictions, test_starts, test_input)
    elapsed = time.perf_counter() - started
    del model
    if device.startswith("cuda"):
        torch.cuda.empty_cache()
    return val_completed, test_completed, elapsed


def verified_classical_shard(archive: zipfile.ZipFile, ticker: str,
                             fold: str, expected_fingerprint: str) -> dict:
    name = f"results/shards/{ticker}_{fold}.csv"
    sidecar = json.loads(archive.read(name.replace(".csv", ".meta.json")))
    blob = archive.read(name)
    if (sidecar["csv_sha256"] != sha256(blob) or
            sidecar["run_fingerprint"] != expected_fingerprint):
        raise ValueError(f"classical checkpoint checksum mismatch: {ticker}/{fold}")
    # Pandas' default parser can turn the archived 0.29999999999999999 into
    # 0.2999999999999999, which does not equal the prespecified Python 0.3.
    frame = pd.read_csv(io.BytesIO(blob), float_precision="round_trip")
    reference = frame.loc[(frame.method == "linear") & (frame.feature == "open")]
    if len(reference) != len(BLOCKS) * len(RATES) * len(SEEDS):
        raise ValueError(f"incomplete classical condition grid: {ticker}/{fold}")
    return {(int(row.block_length), float(row.rate), int(row.seed)):
            (int(row.train_mask_seed), int(row.test_mask_seed))
            for row in reference.itertuples()}


def migrate_checkpoints(output_dir: Path, fingerprint: str,
                        allowed_previous: set[str]) -> dict:
    """Relabel intact shards written under an earlier environment fingerprint.

    This is a one-time migration for checkpoints produced before the resume key
    was made hardware independent. Only a fingerprint listed in
    ``allowed_previous`` is accepted, so a genuine protocol change can never be
    laundered into the current run. Only the provenance key is rewritten. Every
    CSV is re-hashed first, so a shard is kept only when its bytes still match
    the digest recorded beside it. Anything that fails verification is deleted
    and recomputed.
    """
    migrated = 0
    dropped = 0
    superseded: set[str] = set()
    for directory, expected in ((output_dir / "shards", 10),
                                (output_dir / "validation_shards", 35)):
        if not directory.is_dir():
            continue
        for meta_path in sorted(directory.glob("*.meta.json")):
            csv_path = meta_path.with_name(meta_path.name.replace(".meta.json", ".csv"))
            if not csv_path.exists():
                meta_path.unlink(missing_ok=True)
                dropped += 1
                continue
            try:
                sidecar = json.loads(meta_path.read_text())
            except json.JSONDecodeError:
                meta_path.unlink(missing_ok=True)
                csv_path.unlink(missing_ok=True)
                dropped += 1
                continue
            if sidecar.get("run_fingerprint") == fingerprint:
                continue
            if sidecar.get("run_fingerprint") not in allowed_previous:
                continue
            if (sidecar.get("expected_rows") != expected or
                    sidecar.get("csv_sha256") != sha256(csv_path.read_bytes())):
                meta_path.unlink(missing_ok=True)
                csv_path.unlink(missing_ok=True)
                dropped += 1
                continue
            previous = str(sidecar.get("run_fingerprint"))
            superseded.add(previous)
            sidecar.setdefault("migrated_from", []).append(previous)
            sidecar["run_fingerprint"] = fingerprint
            atomic_write_bytes(meta_path, json.dumps(sidecar, indent=2).encode())
            migrated += 1
    return {"migrated": migrated, "dropped": dropped,
            "superseded_fingerprints": sorted(superseded)}


def atomic_write_bytes(path: Path, data: bytes) -> None:
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_bytes(data)
    os.replace(temporary, path)


def run(snapshot: Path, classical_zip: Path, output_dir: Path,
        pilot: bool = False) -> None:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    with zipfile.ZipFile(snapshot) as private, zipfile.ZipFile(classical_zip) as classical:
        manifest_blob = private.read("asset_universe_v1.json")
        receipt_blob = private.read("acquisition_qc_receipt.json")
        manifest_sha = sha256(manifest_blob)
        receipt = json.loads(receipt_blob)
        classical_meta = json.loads(classical.read("results/run_metadata.json"))
        classical_marker = json.loads(classical.read("results/COMPLETE.marker"))
        if (receipt["manifest_sha256"] != manifest_sha or
                classical_meta["manifest_sha256"] != manifest_sha or
                classical_meta["private_zip_sha256"] != sha256(snapshot.read_bytes()) or
                classical_marker["expected_shards"] != 102 or
                classical_marker["pilot_not_publishable"] or
                classical_marker["run_fingerprint"] != classical_meta["run_fingerprint"]):
            raise ValueError("private/classical inputs are incomplete or do not match")
        if (receipt["eligible_count"] != 34 or receipt["excluded_count"] or
                tuple(classical_meta["methods"]) != METHODS or
                classical_meta["edge_guard"] != EDGE_GUARD):
            raise ValueError("classical protocol or panel differs from the frozen run")

        versions = {"python": platform.python_version(),
                    "numpy": np.__version__, "pandas": pd.__version__,
                    "scipy": scipy.__version__, "sklearn": sklearn.__version__,
                    "torch": torch.__version__, "pypots": pypots.__version__,
                    "pygrinder": pygrinder.__version__,
                    "torch_cuda": torch.version.cuda,
                    "cudnn": torch.backends.cudnn.version()}
        hardware = {"device": device,
                    "gpu_name": torch.cuda.get_device_name(0)
                    if device == "cuda" else None}
        config = {"window_length": N_STEPS, "window_stride": STRIDE,
                  "batch_size": BATCH_SIZE, "epochs_max": EPOCHS,
                  "patience": PATIENCE, "brits_hidden": BRITS_HIDDEN,
                  "saits": SAITS_CONFIG, "methods": NEURAL_METHODS,
                  "device": device, "hardware": hardware}
        # The resume key covers the protocol only. Accelerator model, driver
        # stack and framework build are recorded in the metadata but excluded,
        # so a run may legitimately span several sessions and GPUs.
        protocol_config = {"window_length": N_STEPS, "window_stride": STRIDE,
                           "batch_size": BATCH_SIZE, "epochs_max": EPOCHS,
                           "patience": PATIENCE, "brits_hidden": BRITS_HIDDEN,
                           "saits": SAITS_CONFIG, "methods": NEURAL_METHODS}
        protocol_versions = {key: versions[key] for key in
                             ("numpy", "pandas", "scipy", "sklearn",
                              "pypots", "pygrinder")}
        fingerprint = sha256(
            Path(__file__).read_bytes() + Path(__file__).with_name("masking.py").read_bytes()
            + Path(__file__).with_name("run_classical_benchmark.py").read_bytes()
            + manifest_blob + receipt_blob + classical_marker["run_fingerprint"].encode()
            + json.dumps(protocol_config, sort_keys=True).encode()
            + json.dumps(protocol_versions, sort_keys=True).encode()
            + str(pilot).encode())
        output_dir.mkdir(parents=True, exist_ok=True)
        shard_dir = output_dir / "shards"
        shard_dir.mkdir(exist_ok=True)
        validation_dir = output_dir / "validation_shards"
        validation_dir.mkdir(exist_ok=True)
        meta_path = output_dir / "run_metadata.json"
        previous_meta = (json.loads(meta_path.read_text())
                         if meta_path.exists() else {})
        previous_key = previous_meta.get("run_fingerprint")
        legacy = bool(previous_meta) and "resume_key_scope" not in previous_meta
        migration = {"migrated": 0, "dropped": 0, "superseded_fingerprints": []}
        if legacy and previous_key and previous_key != fingerprint:
            # One-time move from the old hardware-dependent key to the new one.
            migration = migrate_checkpoints(output_dir, fingerprint,
                                            {previous_key})
            print(f"Checkpoint migration: {migration['migrated']} relabelled, "
                  f"{migration['dropped']} discarded as unverifiable",
                  flush=True)
        elif previous_key and previous_key != fingerprint:
            changed = {k: (previous_meta.get("protocol_versions", {}).get(k), v)
                       for k, v in protocol_versions.items()
                       if previous_meta.get("protocol_versions", {}).get(k) != v}
            raise SystemExit(
                "Protokol anahtari degisti, tamamlanmis shard'lar bu kosuya "
                "dahil edilemez. Farkli kutuphaneler: " + json.dumps(changed)
                + ". Ya ortami eski surumlere sabitleyin ya da sonuc klasorunu "
                "yeni bir isimle bastan baslatin.")

        history_path = output_dir / "environment_history.json"
        history = (json.loads(history_path.read_text())
                   if history_path.exists() else [])
        history.append({"started_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ",
                                                     time.gmtime()),
                        "versions": versions, "hardware": hardware,
                        "run_fingerprint": fingerprint,
                        "checkpoint_migration": migration})
        atomic_write_bytes(history_path, json.dumps(history, indent=2).encode())

        metadata = {"run_fingerprint": fingerprint,
                    "resume_key_scope": "protocol only; accelerator, driver "
                                        "and framework build excluded",
                    "sessions": len(history),
                    "protocol_versions": protocol_versions,
                    "environment_history": "environment_history.json",
                    "classical_fingerprint": classical_marker["run_fingerprint"],
                    "manifest_sha256": manifest_sha,
                    "private_zip_sha256": classical_meta["private_zip_sha256"],
                    "classical_zip_sha256": sha256(classical_zip.read_bytes()),
                    "pilot_not_publishable": pilot,
                    "model_selection": "separate masked validation partition",
                    "train_and_val_data": "condition-specific masks, no test labels",
                    "window_overlap_aggregation": "arithmetic mean",
                    "config": config, "versions": versions,
                    "hardware": hardware}
        atomic_write_bytes(output_dir / "run_metadata.json",
                           json.dumps(metadata, indent=2).encode())

        records = receipt["assets"][:1] if pilot else receipt["assets"]
        folds = ("A",) if pilot else tuple(FOLDS)
        conditions = [(b, r, s) for b in BLOCKS for r in RATES for s in SEEDS]
        if pilot:
            conditions = conditions[:1]
        expected_shards = len(records) * len(folds) * len(conditions)
        session_started = time.time()
        session_done = 0
        already = sum(1 for _ in shard_dir.glob("*.csv"))
        print(f"Resuming: {already}/{expected_shards} shards already on disk "
              f"({100 * already / expected_shards:.1f}%)", flush=True)
        for asset_index, record in enumerate(records, 1):
            ticker = record["ticker"]
            frame = load_verified_asset(private, record)
            for fold in folds:
                reference = verified_classical_shard(
                    classical, ticker, fold, classical_marker["run_fingerprint"])
                partitions = split_frame(frame, fold)
                train = partitions["train"].to_numpy(float)
                val = partitions["val"].to_numpy(float)
                test = partitions["test"].to_numpy(float)
                for condition_index, (block, rate, repetition_seed) in enumerate(conditions, 1):
                    stem = f"{ticker}_{fold}_b{block}_r{round(rate*100)}_s{repetition_seed}"
                    csv_path = shard_dir / (stem + ".csv")
                    meta_path = shard_dir / (stem + ".meta.json")
                    val_csv_path = validation_dir / (stem + ".csv")
                    val_meta_path = validation_dir / (stem + ".meta.json")
                    if (csv_path.exists() and meta_path.exists() and
                            val_csv_path.exists() and val_meta_path.exists()):
                        sidecar = json.loads(meta_path.read_text())
                        val_sidecar = json.loads(val_meta_path.read_text())
                        if (sidecar.get("run_fingerprint") == fingerprint and
                                sidecar.get("expected_rows") == 10 and
                                sidecar.get("csv_sha256") == sha256(csv_path.read_bytes()) and
                                val_sidecar.get("run_fingerprint") == fingerprint and
                                val_sidecar.get("expected_rows") == 35 and
                                val_sidecar.get("csv_sha256") == sha256(val_csv_path.read_bytes())):
                            continue
                    train_seed = stable_mask_seed(manifest_sha, ticker, fold, "train",
                                                  block, rate, repetition_seed)
                    val_seed = stable_mask_seed(manifest_sha, ticker, fold, "val",
                                                block, rate, repetition_seed)
                    test_seed = stable_mask_seed(manifest_sha, ticker, fold, "test",
                                                 block, rate, repetition_seed)
                    if reference[(block, rate, repetition_seed)] != (train_seed, test_seed):
                        raise AssertionError("neural and classical mask seeds differ")
                    train_hidden = strict_block_mask(train.shape, rate, block,
                                                     train_seed, EDGE_GUARD)
                    val_hidden = strict_block_mask(val.shape, rate, block,
                                                   val_seed, EDGE_GUARD)
                    test_hidden = strict_block_mask(test.shape, rate, block,
                                                    test_seed, EDGE_GUARD)
                    mean, scale = train_observed_scaler(train, train_hidden)
                    train_input = corrupt((train - mean) / scale, train_hidden)
                    val_truth = (val - mean) / scale
                    val_input = corrupt(val_truth, val_hidden)
                    test_input = corrupt((test - mean) / scale, test_hidden)
                    lengths = run_lengths(test_hidden)
                    if int(lengths.max()) > block:
                        raise AssertionError("test mask exceeds requested run length")
                    common = {"ticker": ticker, "country": record["country"],
                              "region": record["region"], "fold": fold,
                              "test_start_year": FOLDS[fold]["test"][0],
                              "test_end_year": FOLDS[fold]["test"][1],
                              "block_length": block, "rate": rate,
                              "seed": repetition_seed,
                              "train_mask_seed": train_seed,
                              "val_mask_seed": val_seed,
                              "test_mask_seed": test_seed,
                              "actual_max_test_run": int(lengths.max())}
                    rows = []
                    val_common = {"ticker": ticker, "country": record["country"],
                                  "region": record["region"], "fold": fold,
                                  "block_length": block, "rate": rate,
                                  "seed": repetition_seed,
                                  "train_mask_seed": train_seed,
                                  "val_mask_seed": val_seed,
                                  "partition": "validation"}
                    val_rows = []
                    for method, (val_estimate, val_elapsed, val_warnings) in run_methods(
                            train_input, val_input, repetition_seed).items():
                        val_rows.extend(score_features(
                            val, val_estimate, val_hidden, mean, scale,
                            {**val_common, "method": method},
                            val_elapsed, val_warnings))
                    for method in NEURAL_METHODS:
                        model_seed = training_seed(manifest_sha, ticker, fold,
                                                   block, rate, repetition_seed, method)
                        val_estimate, estimate, elapsed = fit_and_predict(
                            method, train_input, val_input, val_truth,
                            test_input, model_seed, device,
                            epochs=1 if pilot else EPOCHS)
                        val_rows.extend(score_features(
                            val, val_estimate, val_hidden, mean, scale,
                            {**val_common, "method": method,
                             "model_seed": model_seed}, elapsed, 0))
                        scored = score_features(test, estimate, test_hidden,
                                                mean, scale,
                                                {**common, "method": method,
                                                 "model_seed": model_seed,
                                                 "window_length": N_STEPS,
                                                 "window_stride": STRIDE,
                                                 "epochs_max": 1 if pilot else EPOCHS,
                                                 "patience": 1 if pilot else PATIENCE},
                                                elapsed, 0)
                        rows.extend(scored)
                    if len(rows) != len(NEURAL_METHODS) * len(FEATURES):
                        raise AssertionError("neural condition row count mismatch")
                    if len(val_rows) != (len(METHODS) + len(NEURAL_METHODS)) * len(FEATURES):
                        raise AssertionError("validation condition row count mismatch")
                    val_data = pd.DataFrame(val_rows).to_csv(
                        index=False, float_format="%.17g").encode()
                    atomic_write_bytes(val_csv_path, val_data)
                    val_sidecar = {"run_fingerprint": fingerprint,
                                   "expected_rows": len(val_rows),
                                   "csv_sha256": sha256(val_data),
                                   "pilot_not_publishable": pilot}
                    atomic_write_bytes(val_meta_path,
                                       json.dumps(val_sidecar, indent=2).encode())
                    data = pd.DataFrame(rows).to_csv(
                        index=False, float_format="%.17g").encode()
                    atomic_write_bytes(csv_path, data)
                    sidecar = {"run_fingerprint": fingerprint,
                               "expected_rows": len(rows),
                               "csv_sha256": sha256(data),
                               "pilot_not_publishable": pilot}
                    atomic_write_bytes(meta_path, json.dumps(sidecar, indent=2).encode())
                    session_done += 1
                    if session_done % 10 == 0:
                        rate_s = (time.time() - session_started) / session_done
                        left = expected_shards - already - session_done
                        print(f"[{asset_index}/{len(records)}] {ticker}/{fold} "
                              f"{condition_index}/{len(conditions)} | "
                              f"toplam {already + session_done}/{expected_shards} "
                              f"(%{100 * (already + session_done) / expected_shards:.1f}) | "
                              f"{rate_s:.0f} s/shard | kalan ~{rate_s * left / 3600:.1f} saat",
                              flush=True)

        verified = 0
        for record in records:
            for fold in folds:
                for block, rate, repetition_seed in conditions:
                    stem = (f"{record['ticker']}_{fold}_b{block}_"
                            f"r{round(rate*100)}_s{repetition_seed}")
                    csv_path = shard_dir / (stem + ".csv")
                    sidecar = json.loads((shard_dir / (stem + ".meta.json")).read_text())
                    val_csv_path = validation_dir / (stem + ".csv")
                    val_sidecar = json.loads(
                        (validation_dir / (stem + ".meta.json")).read_text())
                    if (sidecar["run_fingerprint"] != fingerprint or
                            sidecar["csv_sha256"] != sha256(csv_path.read_bytes()) or
                            val_sidecar["run_fingerprint"] != fingerprint or
                            val_sidecar["csv_sha256"] != sha256(val_csv_path.read_bytes())):
                        raise AssertionError(f"invalid neural checkpoint: {stem}")
                    verified += 1
        if verified != expected_shards:
            raise AssertionError("missing neural result shards")
        marker = {"expected_shards": expected_shards,
                  "expected_rows": expected_shards * len(NEURAL_METHODS) * len(FEATURES),
                  "expected_validation_rows": expected_shards *
                  (len(METHODS) + len(NEURAL_METHODS)) * len(FEATURES),
                  "pilot_not_publishable": pilot,
                  "run_fingerprint": fingerprint}
        atomic_write_bytes(output_dir / "COMPLETE.marker",
                           json.dumps(marker, indent=2).encode())
        print("COMPLETE", output_dir, flush=True)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--snapshot", type=Path, required=True)
    parser.add_argument("--classical", type=Path, required=True)
    parser.add_argument("--out", type=Path, required=True)
    parser.add_argument("--pilot", action="store_true",
                        help="one asset/fold/condition and one epoch; not publishable")
    args = parser.parse_args()
    run(args.snapshot, args.classical, args.out, pilot=args.pilot)


In [ ]:
%%writefile /content/itri_neural/run_neural_windownorm.py
#!/usr/bin/env python3
"""EXPLORATORY neural rerun (protocol v2 amendment): per-window instance normalization.

Identical to the frozen v1 run (same private snapshot, folds, masks, training-period
scaling, windows, architecture, seeds, epoch budget for variant V1) with ONE change:
every 64-day window is standardized by the mean/std of its own observed cells before
the network sees it, and predictions are de-standardized with the same statistics
(RevIN-style instance normalization). Variant V2 additionally raises the epoch budget
(200, patience 20). Output goes to a separate directory and is labeled exploratory.
Resumable: one CSV + sidecar per asset/fold/variant/method.
"""
from __future__ import annotations
import argparse, json, os, sys, time, zipfile, hashlib, platform
from pathlib import Path
import numpy as np, pandas as pd, torch
CODE_DIR = os.environ.get("ITRI_CODE_DIR", "results/code_and_protocol")
SNAPSHOT = os.environ.get("ITRI_SNAPSHOT", "private/snapshot.zip")
sys.path.insert(0, CODE_DIR)
from masking import run_lengths, strict_block_mask  # noqa
from run_classical_benchmark import (BLOCKS, EDGE_GUARD, FEATURES, FOLDS, RATES, SEEDS, corrupt,  # noqa
                                     load_verified_asset, score_features, sha256, split_frame,
                                     stable_mask_seed, train_observed_scaler)
from run_neural_benchmark import (N_STEPS, STRIDE, BATCH_SIZE, BRITS_HIDDEN, SAITS_CONFIG,  # noqa
                                  window_starts, as_windows, fold_windows, set_training_seed,
                                  training_seed)
from pypots.imputation import BRITS, SAITS  # noqa
import logging
from pypots.utils.logging import logger as pypots_logger
pypots_logger.setLevel(logging.ERROR)

VARIANTS = {"V1": dict(epochs=30, patience=5), "V2": dict(epochs=200, patience=20)}
PROTOCOL_ID = "ITRI-neural-protocol-v2-exploratory-2026-09-29|winnorm-observed-cells|fallback<2obs-or-sd<1e-6"


def window_stats(win):
    mu = np.nanmean(win, axis=1, keepdims=True); sd = np.nanstd(win, axis=1, keepdims=True)
    n_obs = np.sum(np.isfinite(win), axis=1, keepdims=True)
    bad = (n_obs < 2) | ~np.isfinite(sd) | (sd < 1e-6)
    return np.where(bad, 0.0, mu).astype(np.float32), np.where(bad, 1.0, sd).astype(np.float32), int(bad.sum())


def fit_predict(method, train_in, val_in, val_truth, test_in, seed, epochs, patience, device):
    set_training_seed(seed)
    tr_s, va_s, te_s = window_starts(len(train_in)), window_starts(len(val_in)), window_starts(len(test_in))
    Xtr, Xva, Xva_ori, Xte = (as_windows(a, s) for a, s in ((train_in, tr_s), (val_in, va_s), (val_truth, va_s), (test_in, te_s)))
    mtr, s_tr, f1 = window_stats(Xtr); mva, s_va, f2 = window_stats(Xva); mte, s_te, f3 = window_stats(Xte)
    Xtr = (Xtr - mtr) / s_tr; Xva_ori = (Xva_ori - mva) / s_va; Xva = (Xva - mva) / s_va; Xte = (Xte - mte) / s_te
    common = dict(n_steps=N_STEPS, n_features=len(FEATURES), batch_size=BATCH_SIZE, epochs=epochs,
                  patience=min(patience, epochs), device=device, model_saving_strategy=None, verbose=False)
    model = BRITS(rnn_hidden_size=BRITS_HIDDEN, **common) if method == "brits" else SAITS(**SAITS_CONFIG, **common)
    t0 = time.perf_counter()
    model.fit({"X": Xtr}, {"X": Xva, "X_ori": Xva_ori})
    Pva = np.asarray(model.predict({"X": Xva})["imputation"]) * s_va + mva
    Pte = np.asarray(model.predict({"X": Xte})["imputation"]) * s_te + mte
    el = time.perf_counter() - t0
    del model
    return fold_windows(Pva, va_s, val_in), fold_windows(Pte, te_s, test_in), el, f1 + f2 + f3


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--method", required=True, choices=["saits", "brits"])
    ap.add_argument("--variant", required=True, choices=list(VARIANTS))
    ap.add_argument("--out", default="neural_windownorm")
    ap.add_argument("--workers", type=int, default=1)
    ap.add_argument("--worker-id", type=int, default=0)
    ap.add_argument("--threads", type=int, default=1)
    ap.add_argument("--seeds", default="11,23,37")
    ap.add_argument("--folds", default="A,B,C")
    a = ap.parse_args()
    torch.set_num_threads(a.threads)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    cfg = VARIANTS[a.variant]
    out = Path(a.out); shards = out / "shards"; shards.mkdir(parents=True, exist_ok=True)
    seeds = tuple(int(s) for s in a.seeds.split(",")); folds = tuple(a.folds.split(","))
    with zipfile.ZipFile(SNAPSHOT) as z:
        manifest_blob = z.read("asset_universe_v1.json"); manifest_sha = sha256(manifest_blob)
        receipt = json.loads(z.read("acquisition_qc_receipt.json"))
        records = receipt["assets"]
        # Resume key covers the amendment protocol only (not the runner bytes or hardware).
        fingerprint = sha256(PROTOCOL_ID.encode() + manifest_blob + json.dumps(cfg, sort_keys=True).encode()
                             + a.method.encode() + a.variant.encode())
        meta = {"label": "EXPLORATORY protocol v2 amendment: per-window instance normalization",
                "method": a.method, "variant": a.variant, "config": cfg, "fingerprint": fingerprint,
                "manifest_sha256": manifest_sha, "private_zip_sha256": sha256(Path(SNAPSHOT).read_bytes()),
                "device": device, "gpu": torch.cuda.get_device_name(0) if device == "cuda" else None,
                "versions": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
                             "torch": torch.__version__, "pypots": __import__("pypots").__version__},
                "window_length": N_STEPS, "window_stride": STRIDE, "batch_size": BATCH_SIZE}
        (out / f"run_metadata_{a.method}_{a.variant}.json").write_text(json.dumps(meta, indent=2))
        jobs = [(r, f) for r in records for f in folds]
        jobs = jobs[a.worker_id::a.workers]
        conditions = [(b, r, s) for b in BLOCKS for r in RATES for s in seeds]
        def piece_paths(rec, fold):
            stem = f"{rec['ticker']}_{fold}_{a.method}_{a.variant}"
            return stem, shards / f"{stem}.csv", shards / f"{stem}.meta.json"

        def is_complete(csv_path, meta_path):
            """A piece counts as done only if its sidecar parses and the CSV bytes match the recorded hash."""
            if not (csv_path.exists() and meta_path.exists()):
                return False
            try:
                side = json.loads(meta_path.read_text())
            except (json.JSONDecodeError, OSError):
                return False          # truncated sidecar (e.g. abrupt disconnect): recompute this piece
            return side.get("fingerprint") == fingerprint and side.get("csv_sha256") == sha256(csv_path.read_bytes())

        already = sum(is_complete(*piece_paths(r, f)[1:]) for r, f in jobs)
        print(f"[worker {a.worker_id}] {a.method} {a.variant}: {already}/{len(jobs)} pieces already complete and verified; "
              f"{len(jobs) - already} to compute ({len(conditions)} conditions per piece)", flush=True)
        t_start = time.time(); done = 0
        for rec, fold in jobs:
            ticker = rec["ticker"]
            stem, csv_path, meta_path = piece_paths(rec, fold)
            if is_complete(csv_path, meta_path):
                continue
            frame = load_verified_asset(z, rec)
            parts = split_frame(frame, fold)
            train, val, test = (parts[k].to_numpy(float) for k in ("train", "val", "test"))
            rows = []
            for block, rate, rep in conditions:
                sd = {p: stable_mask_seed(manifest_sha, ticker, fold, p, block, rate, rep) for p in ("train", "val", "test")}
                trh = strict_block_mask(train.shape, rate, block, sd["train"], EDGE_GUARD)
                vah = strict_block_mask(val.shape, rate, block, sd["val"], EDGE_GUARD)
                teh = strict_block_mask(test.shape, rate, block, sd["test"], EDGE_GUARD)
                mean, scale = train_observed_scaler(train, trh)
                train_in = corrupt((train - mean) / scale, trh)
                val_truth = (val - mean) / scale; val_in = corrupt(val_truth, vah)
                test_in = corrupt((test - mean) / scale, teh)
                if int(run_lengths(teh).max()) > block:
                    raise AssertionError("mask invariant")
                mseed = training_seed(manifest_sha, ticker, fold, block, rate, rep, a.method)
                v_est, t_est, el, fb = fit_predict(a.method, train_in, val_in, val_truth, test_in, mseed, device=device, **cfg)
                common = {"ticker": ticker, "country": rec["country"], "region": rec["region"], "fold": fold,
                          "block_length": block, "rate": rate, "seed": rep, "train_mask_seed": sd["train"],
                          "val_mask_seed": sd["val"], "test_mask_seed": sd["test"], "model_seed": mseed,
                          "method": f"{a.method}_winnorm_{a.variant}", "variant": a.variant,
                          "epochs_max": cfg["epochs"], "patience": cfg["patience"], "window_fallbacks": fb}
                rows.extend(score_features(test, t_est, teh, mean, scale, {**common, "partition": "test"}, el, 0))
                rows.extend(score_features(val, v_est, vah, mean, scale, {**common, "partition": "validation"}, el, 0))
            data = pd.DataFrame(rows).to_csv(index=False, float_format="%.17g").encode()
            tmp = csv_path.with_suffix(".csv.tmp"); tmp.write_bytes(data); os.replace(tmp, csv_path)
            mtmp = meta_path.with_name(meta_path.name + ".tmp")
            mtmp.write_text(json.dumps({"fingerprint": fingerprint, "csv_sha256": sha256(data), "rows": len(rows)}, indent=2))
            os.replace(mtmp, meta_path)
            done += 1
            el_total = time.time() - t_start
            total_done = already + done
            left = len(jobs) - total_done
            print(f"[worker {a.worker_id}] {stem} saved | total {total_done}/{len(jobs)} complete "
                  f"({done} this session, {el_total/60:.1f} min) | ~{el_total/done*left/60:.0f} min left for the remaining {left}",
                  flush=True)
    print("worker finished", a.worker_id)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/itri_neural/NEURAL_PROTOCOL_v2_exploratory.md
# ITRI external validation: protocol amendment v2 (EXPLORATORY), 2026-09-29

This amendment was written **after** the complete v1 results (classical
comparators, BRITS, SAITS; 5,508 conditions) had been inspected. Every analysis
under this amendment is therefore exploratory. It does not replace the
prespecified v1 primary result (ITRI `w=1`, `p=5` versus the frozen comparator
set) and must be reported in a separately labeled section.

## What v1 showed and why the amendment exists

- Under v1, BRITS and SAITS reached feature-macro NRMSE of 1.20 and 1.15
  (asset-level means), against 0.27 for linear interpolation. Their error was
  almost flat across gap lengths (1-day gaps were not filled better than
  20-day gaps), rose with the chronological distance between training and test
  periods (fold A 1.6-1.7, fold B 0.75-0.80, fold C 1.05-1.09), and correlated
  with the test-period displacement from the training distribution
  (Spearman 0.96 between the mean absolute training-scaled z-score of test
  prices and neural NRMSE; 0.23 for linear interpolation). KNN (0.97) and the
  chained-equation imputer (0.75) showed the same dependence.
- The v1 feature scaling (training-observed mean and standard deviation) is
  inherited from the classical benchmark. It is level-dependent: any imputer
  whose output depends on the absolute scaled level fails when test prices
  leave the training range. Temporal interpolators are level-invariant.
- A CPU pilot (fold A; THD, EWZ, EWK, EWG, EWT, EDEN; blocks 1, 5, 20; rate
  0.20; seed 11) showed that per-window instance normalization alone reduces
  SAITS test NRMSE from 1.81 to 0.27 at the unchanged 30-epoch budget and to
  0.24 with a 200-epoch budget. Pilot numbers are not reported as results.

## Amendment (the only changes relative to v1)

1. **Per-window instance normalization.** Every 64-day window given to BRITS
   or SAITS (training, validation, test) is standardized per feature by the
   mean and standard deviation of that window's *observed* cells, computed
   after the v1 training-period scaling. Predictions are de-standardized with
   the same window statistics before overlapping windows are averaged and
   observed cells are restored. Hidden cells never enter the statistics. If a
   window has fewer than two observed cells for a feature, or a standard
   deviation below 1e-6, the identity transform is used and the fallback is
   counted. This is the reversible instance normalization of Kim et al.
   (ICLR 2022) applied without learnable affine parameters.
2. **Two training budgets.** Variant V1 keeps the v1 budget exactly (30
   epochs, patience 5) so that the normalization effect is isolated. Variant
   V2 raises the budget to 200 epochs with patience 20. Architecture, window
   length, stride, batch size, optimizer defaults, seeds and mask sets are
   unchanged.
3. **Scope.** SAITS is rerun on the full 5,508-condition grid for V1 and V2.
   BRITS is rerun for V1 on the full grid when compute allows, otherwise on a
   labeled subset; the subset is stated with the results.
4. **Reporting.** Results are reported beside the v1 results with the
   exploratory label. The v1 numbers are not removed or replaced. No further
   architecture or hyperparameter search is performed after inspecting the
   amended scores. The same asset-level unit of analysis applies; no
   inferential test is added.
5. **Provenance.** Each run writes its own fingerprint (runner code, manifest,
   configuration, method, variant), library versions and hardware. Bitwise
   equality across hardware is not claimed; the pilot reproduced the archived
   v1 BRITS score for one condition to three decimals on CPU, and SAITS to
   within 0.05-0.20 NRMSE, which is the expected nondeterminism.

## Interpretation rules fixed before the amended results were seen

- If amended SAITS or BRITS beat ITRI on the external panel, the manuscript
  reports this as the primary finding of the amendment; ITRI's claim is
  restricted to the regimes where it retains an advantage (short gaps,
  single-feature dependence, negligible cost).
- If they do not, the manuscript still reports the level-shift mechanism and
  the recovery, without claiming ITRI superiority over properly normalized
  deep imputers in general.

## Status (2026-09-29, after execution)

- SAITS V1 (30 epochs, per-window normalization): completed on the full
  5,508-condition grid on a two-core CPU environment (PyPOTS 1.5, torch 2.14,
  median 6.2 s per condition, zero normalization fallbacks). Reported in the
  manuscript as Table 20 and Figures 5(b) and 7.
- SAITS V2 (200 epochs) and BRITS V1: not completed at full scale within the
  study. Any subset that is completed is stored in `neural_windownorm_subset/`
  with its own metadata and is labeled by fold and seed; it is not merged into
  the full-grid tables. The Colab notebook `ITRI_neural_windownorm_Colab.ipynb`
  resumes both runs.
- The CPU pilot values that motivated the amendment are excluded from all
  manuscript tables, as required above.


## Ozel veri dosyasi

Sadece **`ITRI_external_PRIVATE_data_snapshot.zip`** dosyasini secin.

In [ ]:
import hashlib
from google.colab import files
print('Select ITRI_external_PRIVATE_data_snapshot.zip only.', flush=True)
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Select exactly one file')
filename, content = next(iter(uploaded.items()))
if hashlib.sha256(content).hexdigest() != '48dc63654f63db3ef9d6d5db7ab69ad502aa079e9f6deb4b276cf9b1170f324b':
    raise ValueError(f'Wrong file: {filename}')
snapshot_path = work_dir / 'PRIVATE_vendor_snapshot.zip'
snapshot_path.write_bytes(content)
del uploaded, content
print('Private snapshot verified.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
result_dir = Path('/content/drive/MyDrive/ITRI_neural_windownorm')
result_dir.mkdir(parents=True, exist_ok=True)
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (GPU runtime secin)')
done = len(list((result_dir / 'shards').glob('*_saits_V2.csv'))) if (result_dir / 'shards').is_dir() else 0
sub = result_dir / 'subset_foldA_seed11' / 'shards'
done_sub = len(list(sub.glob('*_brits_V1.csv'))) if sub.is_dir() else 0
print(f"Drive'da tamamlanmis parca: SAITS V2 {done}/102 varlik-fold, BRITS V1 alt kume {done_sub}/34 varlik")

In [ ]:
import os, subprocess, sys
env = dict(os.environ, ITRI_CODE_DIR=str(work_dir), ITRI_SNAPSHOT=str(snapshot_path))
# (method, variant, extra args). SAITS V1 is deliberately absent: completed on CPU, reported in the manuscript.
RUNS = [
    ('saits', 'V2', []),                                   # full grid, 5,508 conditions
    ('brits', 'V1', ['--seeds', '11', '--folds', 'A']),    # labeled subset: fold A, seed 11, 612 conditions
]
for method, variant, extra in RUNS:
    print(f'=== {method} {variant} {extra} ===', flush=True)
    out_dir = result_dir if not extra else result_dir / 'subset_foldA_seed11'
    cmd = [sys.executable, str(work_dir / 'run_neural_windownorm.py'), '--method', method, '--variant', variant,
           '--out', str(out_dir), '--workers', '1', '--worker-id', '0', '--threads', '2'] + extra
    r = subprocess.run(cmd, cwd=str(work_dir), env=env)
    if r.returncode != 0:
        raise SystemExit(f'{method} {variant} failed with code {r.returncode}; Run all again to resume')

In [ ]:
import zipfile
result_zip = Path('/content/ITRI_neural_windownorm_results.zip')
with zipfile.ZipFile(result_zip, 'w', compression=zipfile.ZIP_DEFLATED) as out:
    for path in result_dir.rglob('*'):
        if path.is_file():
            out.write(path, 'neural_windownorm/' + str(path.relative_to(result_dir)))
    for name in ('masking.py', 'run_classical_benchmark.py', 'run_neural_benchmark.py',
                 'run_neural_windownorm.py', 'NEURAL_PROTOCOL_v2_exploratory.md'):
        out.write(work_dir / name, 'code_and_protocol/' + name)
print('Complete:', result_zip.name, '- downloading now')
files.download(str(result_zip))